In [1]:
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, train_test_split
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.load_data_BCICIV import load_all_subjects
from src.preprocess import laplacian_filter
from src.preprocess import normalize_trial
from src.train_EEGNet import train_model, evaluate_model
from models.EEGNet import EEGNet
from torch.utils.data import DataLoader, Dataset, Subset
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)


# 22 channels (raw + normalization)

In [2]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path, channels_to_use='all')

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [3]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [4]:
split_index_test = 1150
split_index_val = 862 

dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [5]:
def init_weights_xavier(m):
    if isinstance(m, nn.Conv2d):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
            
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [6]:
model = EEGNet(22, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_22_raw = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7196 | Train Acc 0.5104 | Val Loss 0.6904 | Val Acc 0.5451
Epoch 2: Train Loss 0.7063 | Train Acc 0.5081 | Val Loss 0.6797 | Val Acc 0.5521
Epoch 3: Train Loss 0.6917 | Train Acc 0.5394 | Val Loss 0.6750 | Val Acc 0.5625
Epoch 4: Train Loss 0.6697 | Train Acc 0.5870 | Val Loss 0.6712 | Val Acc 0.5903
Epoch 5: Train Loss 0.6617 | Train Acc 0.6044 | Val Loss 0.6722 | Val Acc 0.5729
Epoch 6: Train Loss 0.6549 | Train Acc 0.6090 | Val Loss 0.6647 | Val Acc 0.6076
Epoch 7: Train Loss 0.6421 | Train Acc 0.6323 | Val Loss 0.6530 | Val Acc 0.6562
Epoch 8: Train Loss 0.6269 | Train Acc 0.6671 | Val Loss 0.6489 | Val Acc 0.6458
Epoch 9: Train Loss 0.6133 | Train Acc 0.6624 | Val Loss 0.6377 | Val Acc 0.6667
Epoch 10: Train Loss 0.5892 | Train Acc 0.7216 | Val Loss 0.6305 | Val Acc 0.6562
Epoch 11: Train Loss 0.5918 | Train Acc 0.6937 | Val Loss 0.6238 | Val Acc 0.6667
Epoch 12: Train Loss 0.5881 | Train Acc 0.7007 | Val Loss 0.6198 | Val Acc 0.6632
Epoch 13: Train Loss 0.57

In [7]:
test_acc = evaluate_model(trained_model_22_raw, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.5278


# 22 channels + laplacian filter on C3 and C4

In [8]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [9]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(22, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_22_lap = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7189 | Train Acc 0.4733 | Val Loss 0.6932 | Val Acc 0.4965
Epoch 2: Train Loss 0.6995 | Train Acc 0.5186 | Val Loss 0.6962 | Val Acc 0.4757
Epoch 3: Train Loss 0.6912 | Train Acc 0.5371 | Val Loss 0.6956 | Val Acc 0.5278
Epoch 4: Train Loss 0.6746 | Train Acc 0.5916 | Val Loss 0.6951 | Val Acc 0.5174
Epoch 5: Train Loss 0.6616 | Train Acc 0.6090 | Val Loss 0.6925 | Val Acc 0.5069
Epoch 6: Train Loss 0.6574 | Train Acc 0.6021 | Val Loss 0.6923 | Val Acc 0.5035
Epoch 7: Train Loss 0.6533 | Train Acc 0.6265 | Val Loss 0.6910 | Val Acc 0.5104
Epoch 8: Train Loss 0.6417 | Train Acc 0.6311 | Val Loss 0.6871 | Val Acc 0.5451
Epoch 9: Train Loss 0.6398 | Train Acc 0.6439 | Val Loss 0.6815 | Val Acc 0.5590
Epoch 10: Train Loss 0.6279 | Train Acc 0.6624 | Val Loss 0.6736 | Val Acc 0.5451
Epoch 11: Train Loss 0.6253 | Train Acc 0.6601 | Val Loss 0.6627 | Val Acc 0.5764
Epoch 12: Train Loss 0.6002 | Train Acc 0.6937 | Val Loss 0.6504 | Val Acc 0.6076
Epoch 13: Train Loss 0.58

In [12]:
test_acc = evaluate_model(trained_model_22_lap, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.5417


# 22 channels - only mu band

In [13]:
data = load_all_subjects(data_path, use_multiband=True, bands=[(8, 12), (13, 30)], channels_to_use='all')

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/

In [14]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = x[0]  # Select the mu band (8-12 Hz)

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [15]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(22, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_22_mu = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7271 | Train Acc 0.5151 | Val Loss 0.6923 | Val Acc 0.5278
Epoch 2: Train Loss 0.6962 | Train Acc 0.5336 | Val Loss 0.6880 | Val Acc 0.5382
Epoch 3: Train Loss 0.6813 | Train Acc 0.5615 | Val Loss 0.6863 | Val Acc 0.5174
Epoch 4: Train Loss 0.6652 | Train Acc 0.5824 | Val Loss 0.6841 | Val Acc 0.5347
Epoch 5: Train Loss 0.6588 | Train Acc 0.5974 | Val Loss 0.6822 | Val Acc 0.5521
Epoch 6: Train Loss 0.6540 | Train Acc 0.6056 | Val Loss 0.6812 | Val Acc 0.5486
Epoch 7: Train Loss 0.6378 | Train Acc 0.6276 | Val Loss 0.6805 | Val Acc 0.5451
Epoch 8: Train Loss 0.6294 | Train Acc 0.6485 | Val Loss 0.6745 | Val Acc 0.5486
Epoch 9: Train Loss 0.6163 | Train Acc 0.6682 | Val Loss 0.6737 | Val Acc 0.5451
Epoch 10: Train Loss 0.6156 | Train Acc 0.6543 | Val Loss 0.6691 | Val Acc 0.5764
Epoch 11: Train Loss 0.5965 | Train Acc 0.6984 | Val Loss 0.6621 | Val Acc 0.5868
Epoch 12: Train Loss 0.5935 | Train Acc 0.6891 | Val Loss 0.6565 | Val Acc 0.6181
Epoch 13: Train Loss 0.58

In [ ]:
test_acc = evaluate_model(trained_model_22_mu, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 22 channels - mu + beta band

In [14]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [7,11] # C3 and C4
        self.c3_neighbours = [1,6,8,13]
        self.c4_neighbours = [5,10,12,17]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        bands, channels, samples = x.shape

        processed_bands = []
        for band in range(bands):
            band_data = x[band]
            band_data = np.expand_dims(band_data, axis=0)
            band_data = laplacian_filter(band_data, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])
            band_data = band_data.squeeze(0)
            processed_bands.append(band_data)

        x = np.concatenate(processed_bands, axis=0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [15]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(44, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_22_mu_beta = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7080 | Train Acc 0.5035 | Val Loss 0.6900 | Val Acc 0.5660
Epoch 2: Train Loss 0.6969 | Train Acc 0.5058 | Val Loss 0.6869 | Val Acc 0.5625
Epoch 3: Train Loss 0.6894 | Train Acc 0.5278 | Val Loss 0.6841 | Val Acc 0.5625
Epoch 4: Train Loss 0.6839 | Train Acc 0.5650 | Val Loss 0.6825 | Val Acc 0.5833
Epoch 5: Train Loss 0.6814 | Train Acc 0.5615 | Val Loss 0.6806 | Val Acc 0.5694
Epoch 6: Train Loss 0.6816 | Train Acc 0.5534 | Val Loss 0.6797 | Val Acc 0.5417
Epoch 7: Train Loss 0.6729 | Train Acc 0.5858 | Val Loss 0.6788 | Val Acc 0.5556
Epoch 8: Train Loss 0.6655 | Train Acc 0.6148 | Val Loss 0.6773 | Val Acc 0.5660
Epoch 9: Train Loss 0.6621 | Train Acc 0.6125 | Val Loss 0.6761 | Val Acc 0.5451
Epoch 10: Train Loss 0.6637 | Train Acc 0.6137 | Val Loss 0.6750 | Val Acc 0.5556
Epoch 11: Train Loss 0.6531 | Train Acc 0.6439 | Val Loss 0.6735 | Val Acc 0.5486
Epoch 12: Train Loss 0.6515 | Train Acc 0.6276 | Val Loss 0.6731 | Val Acc 0.5625
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_22_mu_beta, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels (raw + normalization)

In [2]:
data_path = '../datasets/BCICIV_2a_gdf'

data = load_all_subjects(data_path)

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A01T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A02T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


...

Loaded file: ../datasets/BCICIV_2a_gdf/A07T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A08T.gdf


/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Loaded file: ../datasets/BCICIV_2a_gdf/A09T.gdf


In [3]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [6]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(11, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_11_raw = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7041 | Train Acc 0.5220 | Val Loss 0.6910 | Val Acc 0.5035
Epoch 2: Train Loss 0.6718 | Train Acc 0.5673 | Val Loss 0.6772 | Val Acc 0.5486
Epoch 3: Train Loss 0.6335 | Train Acc 0.6497 | Val Loss 0.6686 | Val Acc 0.5903
Epoch 4: Train Loss 0.6204 | Train Acc 0.6717 | Val Loss 0.6613 | Val Acc 0.5903
Epoch 5: Train Loss 0.5850 | Train Acc 0.7332 | Val Loss 0.6462 | Val Acc 0.6354
Epoch 6: Train Loss 0.5716 | Train Acc 0.7320 | Val Loss 0.6305 | Val Acc 0.6458
Epoch 7: Train Loss 0.5259 | Train Acc 0.7854 | Val Loss 0.6175 | Val Acc 0.6458
Epoch 8: Train Loss 0.5101 | Train Acc 0.7738 | Val Loss 0.5939 | Val Acc 0.6736
Epoch 9: Train Loss 0.4864 | Train Acc 0.7831 | Val Loss 0.5802 | Val Acc 0.6875
Epoch 10: Train Loss 0.4652 | Train Acc 0.8179 | Val Loss 0.5676 | Val Acc 0.6910
Epoch 11: Train Loss 0.4540 | Train Acc 0.8005 | Val Loss 0.5649 | Val Acc 0.7188
Epoch 12: Train Loss 0.4193 | Train Acc 0.8469 | Val Loss 0.5602 | Val Acc 0.7014
Epoch 13: Train Loss 0.40

In [10]:
test_acc = evaluate_model(trained_model_11_raw, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.5139


# 11 channels + laplacian filter on C3 and C4

In [20]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [21]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(11, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_11_lap = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7052 | Train Acc 0.5000 | Val Loss 0.6891 | Val Acc 0.5590
Epoch 2: Train Loss 0.6836 | Train Acc 0.5371 | Val Loss 0.6779 | Val Acc 0.5799
Epoch 3: Train Loss 0.6690 | Train Acc 0.5754 | Val Loss 0.6721 | Val Acc 0.5833
Epoch 4: Train Loss 0.6572 | Train Acc 0.6137 | Val Loss 0.6645 | Val Acc 0.5903
Epoch 5: Train Loss 0.6482 | Train Acc 0.6160 | Val Loss 0.6582 | Val Acc 0.5972
Epoch 6: Train Loss 0.6303 | Train Acc 0.6543 | Val Loss 0.6491 | Val Acc 0.6424
Epoch 7: Train Loss 0.6172 | Train Acc 0.6833 | Val Loss 0.6412 | Val Acc 0.6319
Epoch 8: Train Loss 0.6059 | Train Acc 0.7100 | Val Loss 0.6310 | Val Acc 0.6493
Epoch 9: Train Loss 0.5946 | Train Acc 0.6821 | Val Loss 0.6236 | Val Acc 0.6354
Epoch 10: Train Loss 0.5753 | Train Acc 0.7007 | Val Loss 0.6147 | Val Acc 0.6562
Epoch 11: Train Loss 0.5676 | Train Acc 0.7100 | Val Loss 0.6134 | Val Acc 0.6597
Epoch 12: Train Loss 0.5625 | Train Acc 0.7135 | Val Loss 0.6041 | Val Acc 0.6632
Epoch 13: Train Loss 0.55

In [24]:
test_acc = evaluate_model(trained_model_11_lap, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

Test Accuracy: 0.5417


# 11 channels - only mu band

In [29]:
data = load_all_subjects(data_path, use_multiband=True, bands=[(8, 12), (13, 30)])

/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/envs/py311ml/lib/python3.11/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)
/opt/anaconda3/

In [30]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        x = x[0]  # Select the mu band (8-12 Hz)

        x = np.expand_dims(x, axis=0)

        x = laplacian_filter(x, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])

        x = x.squeeze(0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [31]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(11, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_11_mu = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7108 | Train Acc 0.4861 | Val Loss 0.6996 | Val Acc 0.5278
Epoch 2: Train Loss 0.6931 | Train Acc 0.5360 | Val Loss 0.6963 | Val Acc 0.5243
Epoch 3: Train Loss 0.6956 | Train Acc 0.5186 | Val Loss 0.6949 | Val Acc 0.5417
Epoch 4: Train Loss 0.6712 | Train Acc 0.5963 | Val Loss 0.6932 | Val Acc 0.5660
Epoch 5: Train Loss 0.6755 | Train Acc 0.5940 | Val Loss 0.6915 | Val Acc 0.5590
Epoch 6: Train Loss 0.6641 | Train Acc 0.6090 | Val Loss 0.6897 | Val Acc 0.5590
Epoch 7: Train Loss 0.6565 | Train Acc 0.6334 | Val Loss 0.6883 | Val Acc 0.5590
Epoch 8: Train Loss 0.6598 | Train Acc 0.6125 | Val Loss 0.6861 | Val Acc 0.5556
Epoch 9: Train Loss 0.6394 | Train Acc 0.6636 | Val Loss 0.6846 | Val Acc 0.5660
Epoch 10: Train Loss 0.6463 | Train Acc 0.6659 | Val Loss 0.6831 | Val Acc 0.5625
Epoch 11: Train Loss 0.6392 | Train Acc 0.6740 | Val Loss 0.6814 | Val Acc 0.5590
Epoch 12: Train Loss 0.6204 | Train Acc 0.6937 | Val Loss 0.6786 | Val Acc 0.5799
Epoch 13: Train Loss 0.62

In [ ]:
test_acc = evaluate_model(trained_model_11_mu, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')

# 11 channels - mu + beta band

In [33]:
class EEGDataset(Dataset):
    def __init__(self, X, y, transform=None):
        self.X = X
        self.y = y
        self.transform = transform
        self.channels_laplace = [3,7] # C3 and C4
        self.c3_neighbours = [0,2,4,9]
        self.c4_neighbours = [1,6,8,10]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]

        bands, channels, samples = x.shape

        processed_bands = []
        for band in range(bands):
            band_data = x[band]
            band_data = np.expand_dims(band_data, axis=0)
            band_data = laplacian_filter(band_data, self.channels_laplace, [self.c3_neighbours, self.c4_neighbours])
            band_data = band_data.squeeze(0)
            processed_bands.append(band_data)

        x = np.concatenate(processed_bands, axis=0)

        x = normalize_trial(x)
        x = torch.tensor(x, dtype=torch.float32)
        
        y = torch.tensor(self.y[idx], dtype=torch.long)
        if self.transform:
            x = self.transform(x)
        
        return x, y


In [34]:
dataset = EEGDataset(data['X'], data['y'])
train_EEG = Subset(dataset, indices=range(0, split_index_val))
val_EEG = Subset(dataset, indices=range(split_index_val, split_index_test))
test_EEG = Subset(dataset, indices=range(split_index_test, len(dataset)))

train_dl = DataLoader(train_EEG, batch_size=32, shuffle=True)
val_dl = DataLoader(val_EEG, batch_size=32, shuffle=False)
test_dl = DataLoader(test_EEG, batch_size=32, shuffle=False)

In [ ]:
model = EEGNet(22, 1000, 2, f1=32, D=4, dropout_rate=0.4)
model.apply(init_weights_xavier)

trained_model_11_mu_beta = train_model(model, train_dl, val_dl, epochs=100, lr=0.0003, patience=20)

Epoch 1: Train Loss 0.7033 | Train Acc 0.4814 | Val Loss 0.6995 | Val Acc 0.4826
Epoch 2: Train Loss 0.6932 | Train Acc 0.5232 | Val Loss 0.6964 | Val Acc 0.5035
Epoch 3: Train Loss 0.6901 | Train Acc 0.5534 | Val Loss 0.6943 | Val Acc 0.5069
Epoch 4: Train Loss 0.6830 | Train Acc 0.5441 | Val Loss 0.6921 | Val Acc 0.5243
Epoch 5: Train Loss 0.6760 | Train Acc 0.5905 | Val Loss 0.6899 | Val Acc 0.5278
Epoch 6: Train Loss 0.6645 | Train Acc 0.6148 | Val Loss 0.6875 | Val Acc 0.5451
Epoch 7: Train Loss 0.6704 | Train Acc 0.5905 | Val Loss 0.6857 | Val Acc 0.5382
Epoch 8: Train Loss 0.6657 | Train Acc 0.6056 | Val Loss 0.6832 | Val Acc 0.5451
Epoch 9: Train Loss 0.6629 | Train Acc 0.6172 | Val Loss 0.6825 | Val Acc 0.5382
Epoch 10: Train Loss 0.6545 | Train Acc 0.6230 | Val Loss 0.6802 | Val Acc 0.5312
Epoch 11: Train Loss 0.6474 | Train Acc 0.6613 | Val Loss 0.6785 | Val Acc 0.5451
Epoch 12: Train Loss 0.6377 | Train Acc 0.6589 | Val Loss 0.6772 | Val Acc 0.5694
Epoch 13: Train Loss 0.64

In [ ]:
test_acc = evaluate_model(trained_model_11_mu_beta, test_dl)
print(f'Test Accuracy: {test_acc:.4f}')